In [39]:
import torch
import rasterio
import numpy as np
from torch.utils.data import Dataset
import os
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader


In [40]:
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["VECLIB_MAXIMUM_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

In [41]:
root_csv = "../../../data/challenge2026MIASHS/GLC25_PA_metadata_train.csv"
root_images = "../../../data/challenge2026MIASHS/SatelitePatches/PA-train" 
root_images_test = "../../../data/challenge2026MIASHS/SatelitePatches/PA-test"

In [42]:
# Chargement du fichier metadata
df = pd.read_csv(root_csv)
print(df.head(10))

        lon        lat  year  geoUncertaintyInM  areaInM2         region  \
0  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
1  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
2  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
3  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
4  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
5  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
6  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
7  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
8  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
9  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   

  country  speciesId  surveyId  
0  France     6874.0       212  
1  France      476.0       212  
2  France    11157.0       212  
3  France     8784.0       212 

In [43]:
# On crée un identifiant de zone unique basé sur la proximité géographique
# En arrondissant à 1 décimale, on crée des blocs d'environ 11km x 8km (selon la latitude)
df['zone_id'] = df.apply(lambda x: f"{round(x['lon'], 1)}_{round(x['lat'], 1)}", axis=1)

# On ne garde qu'une ligne par surveyId pour le split (car un surveyId peut avoir plusieurs espèces)
surveys = df[['surveyId', 'zone_id']].drop_duplicates()

In [44]:
print(df.head())

        lon        lat  year  geoUncertaintyInM  areaInM2         region  \
0  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
1  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
2  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
3  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   
4  3.099038  43.134956  2021                5.0     100.0  MEDITERRANEAN   

  country  speciesId  surveyId   zone_id  
0  France     6874.0       212  3.1_43.1  
1  France      476.0       212  3.1_43.1  
2  France    11157.0       212  3.1_43.1  
3  France     8784.0       212  3.1_43.1  
4  France     4530.0       212  3.1_43.1  


In [45]:
train_surveys = []
test_surveys = []

for zone in surveys['zone_id'].unique():
    zone_data = surveys[surveys['zone_id'] == zone]
    
    if len(zone_data) >= 2:
        # Split classique si on a assez de données
        tr, te = train_test_split(zone_data['surveyId'], test_size=0.2, random_state=42)
        train_surveys.extend(tr)
        test_surveys.extend(te)
    else:
        # Si une zone n'a qu'un seul surveyId, on l'envoie par défaut en train
        train_surveys.extend(zone_data['surveyId'])

# Création des DataFrames finaux
df_train = df[df['surveyId'].isin(train_surveys)]
df_test = df[df['surveyId'].isin(test_surveys)]

print(f"Nombre de surveys en Train: {len(train_surveys)}")
print(f"Nombre de surveys en Test: {len(test_surveys)}")

Nombre de surveys en Train: 69616
Nombre de surveys en Test: 19371


In [46]:
def check_file_exists(row):
    # On utilise la même logique que ton _get_path
    s_id = str(row['surveyId'])
    if len(s_id) >= 4:
        path = os.path.join(root_images, s_id[-2:], s_id[-4:-2], f"{s_id}.tiff")
    else:
        path = os.path.join(root_images, s_id, f"{s_id}.tiff")
    return os.path.exists(path)

print("Filtrage des données existantes (cela peut prendre un moment)...")
# On crée un masque pour ne garder que les fichiers présents
mask = df.apply(check_file_exists, axis=1)
df = df[mask].reset_index(drop=True)

print(f"Nombre d'images trouvées : {len(df)}")

Filtrage des données existantes (cela peut prendre un moment)...


Nombre d'images trouvées : 1483191


In [53]:
# 1. Charger Wide ResNet 50-2 avec les poids pré-entraînés
# Ce modèle est plus performant pour capturer des textures riches (comme la végétation)
model = models.wide_resnet50_2(weights='DEFAULT')

# 2. Modifier la première couche (conv1) pour accepter 4 canaux (RGB + NIR)
original_conv = model.conv1
model.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)

# 3. Initialisation du 4ème canal (NIR)
with torch.no_grad():
    # Copie les poids RGB existants
    model.conv1.weight[:, :3, :, :] = original_conv.weight
    # Initialise le canal NIR avec la moyenne des poids RGB pour ne pas partir de zéro
    model.conv1.weight[:, 3, :, :] = original_conv.weight.mean(dim=1)

# 4. Modifier la dernière couche (FC) pour le nombre spécifique d'espèces
num_species = df['speciesId'].nunique()
model.fc = nn.Linear(model.fc.in_features, num_species)

# Déplacement sur le GPU
model = model.to(device)

# 5. Optimisation : On baisse un peu le Learning Rate pour le Wide ResNet
# Un LR trop haut (0.001) sur un modèle large peut causer l'instabilité observée (F1 qui stagne)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

Downloading: "https://download.pytorch.org/models/wide_resnet50_2-9ba9bcbe.pth" to /home/grp3/.cache/torch/hub/checkpoints/wide_resnet50_2-9ba9bcbe.pth


100%|██████████| 263M/263M [00:02<00:00, 115MB/s]  


In [48]:
class SatelliteDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform
        # Mapping des espèces pour avoir des labels entre 0 et N-1
        self.label_map = {val: i for i, val in enumerate(self.df['speciesId'].unique())}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        survey_id = row['surveyId']
        
        # Récupération du chemin selon ta règle
        s_id = str(int(survey_id))
        path = os.path.join(self.root_dir, s_id[-2:], s_id[-4:-2], f"{s_id}.tiff")
        
        # Lecture des 4 bandes avec rasterio
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)  # Shape: (4, 64, 64)
            
        # Normalisation simple (à adapter selon les valeurs de tes pixels)
        img /= 10000.0 
        
        label = self.label_map[row['speciesId']]
        
        return torch.tensor(img), torch.tensor(label)

In [54]:
# Création des instances de Dataset
train_dataset = SatelliteDataset(df_train, root_images)
test_dataset = SatelliteDataset(df_test, root_images)

# Création des DataLoaders
# shuffle=True pour le train est crucial pour que le modèle ne mémorise pas l'ordre
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)

In [55]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [56]:
# !pip install torchmetrics
from torchmetrics.classification import MulticlassF1Score

# Initialisation de la métrique (F1 micro est la valeur par défaut pour Multiclass)
f1_metric = MulticlassF1Score(num_classes=num_species, average='micro').to(device)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    # --- PHASE D'ENTRAÎNEMENT ---
    model.train()
    train_loss = 0.0
    f1_metric.reset()
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Calcul de la F1 pour le batch actuel
        preds = torch.argmax(outputs, dim=1)
        f1_metric.update(preds, labels)
        
        # Affichage tous les 50 batches (pour ne pas polluer la console)
        if batch_idx % 50 == 0:
            current_f1 = f1_metric.compute()
            print(f"Epoch {epoch+1} [{batch_idx}/{len(train_loader)}] "
                  f"| Loss: {loss.item():.4f} | F1 Micro: {current_f1:.4f}")

    # Moyenne finale entraînement
    train_f1_epoch = f1_metric.compute()

    # --- PHASE DE VALIDATION ---
    model.eval()
    test_loss = 0.0
    f1_metric.reset() 
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            f1_metric.update(preds, labels)

    test_f1_epoch = f1_metric.compute()
    
    print(f"\n" + "="*30)
    print(f"FIN EPOCH {epoch+1}")
    print(f"TRAIN -> Loss: {train_loss/len(train_loader):.4f}, F1: {train_f1_epoch:.4f}")
    print(f"TEST  -> Loss: {test_loss/len(test_loader):.4f}, F1: {test_f1_epoch:.4f}")
    print("="*30 + "\n")

Epoch 1 [0/9014] | Loss: 8.5731 | F1 Micro: 0.0000
Epoch 1 [50/9014] | Loss: 6.3663 | F1 Micro: 0.0132
Epoch 1 [100/9014] | Loss: 6.1103 | F1 Micro: 0.0154
Epoch 1 [150/9014] | Loss: 6.2851 | F1 Micro: 0.0176
Epoch 1 [200/9014] | Loss: 6.0377 | F1 Micro: 0.0184
Epoch 1 [250/9014] | Loss: 5.9000 | F1 Micro: 0.0191
Epoch 1 [300/9014] | Loss: 5.7282 | F1 Micro: 0.0198
Epoch 1 [350/9014] | Loss: 5.9575 | F1 Micro: 0.0208
Epoch 1 [400/9014] | Loss: 5.9013 | F1 Micro: 0.0218
Epoch 1 [450/9014] | Loss: 5.7604 | F1 Micro: 0.0223
Epoch 1 [500/9014] | Loss: 5.9465 | F1 Micro: 0.0230
Epoch 1 [550/9014] | Loss: 5.5646 | F1 Micro: 0.0233
Epoch 1 [600/9014] | Loss: 5.4909 | F1 Micro: 0.0238
Epoch 1 [650/9014] | Loss: 5.5572 | F1 Micro: 0.0243
Epoch 1 [700/9014] | Loss: 5.5131 | F1 Micro: 0.0248
Epoch 1 [750/9014] | Loss: 5.7430 | F1 Micro: 0.0252
Epoch 1 [800/9014] | Loss: 5.5910 | F1 Micro: 0.0254
Epoch 1 [850/9014] | Loss: 5.4718 | F1 Micro: 0.0256
Epoch 1 [900/9014] | Loss: 5.6335 | F1 Micro: 0.0

In [ ]:
root_test_csv = "../../../data/challenge2026MIASHS/GLC25_PA_metadata_test.csv"
df_test_final = pd.read_csv(root_test_csv)
root_images_test = "../../../data/challenge2026MIASHS/SatelitePatches/PA-test"

# On récupère l'inverse de ton label_map pour retrouver les vrais IDs d'espèces
# Ton label_map actuel est {speciesId: index}, on veut {index: speciesId}
inv_label_map = {v: k for k, v in train_dataset.label_map.items()}

In [ ]:
import torch.nn.functional as F

def predict_test(model, df_test, root_dir, k=3):
    model.eval()
    results = []
    
    print(f"Lancement des prédictions sur {len(df_test)} surveys...")
    
    with torch.no_grad():
        for _, row in df_test.iterrows():
            # Conversion explicite en int pour enlever le .0
            survey_id = int(row['surveyId'])
            
            # Reconstruction du chemin
            s_id = str(survey_id)
            if len(s_id) >= 4:
                path = os.path.join(root_dir, s_id[-2:], s_id[-4:-2], f"{s_id}.tiff")
            else:
                path = os.path.join(root_dir, s_id, f"{s_id}.tiff")
            
            if not os.path.exists(path):
                results.append({"surveyId": survey_id, "predictions": ""})
                continue

            with rasterio.open(path) as src:
                img = src.read().astype(np.float32) / 10000.0
            
            img_tensor = torch.tensor(img).unsqueeze(0).to(device)
            outputs = model(img_tensor)
            
            probs = F.softmax(outputs, dim=1)
            top_probs, top_indices = torch.topk(probs, k=k)
            
            # Conversion des indices -> speciesId -> entier -> string
            predicted_species = [str(int(inv_label_map[idx.item()])) for idx in top_indices[0]]
            
            results.append({
                "surveyId": survey_id, 
                "predictions": " ".join(predicted_species)
            })
            
    return pd.DataFrame(results)

In [ ]:
# Lancer la prédiction
df_submission = predict_test(model, df_test_final, root_images_test, k=3)

# Sécurité supplémentaire : s'assurer que surveyId est bien typé entier
df_submission['surveyId'] = df_submission['surveyId'].astype(int)

# Sauvegarde sans index et avec séparateur virgule
df_submission.to_csv("submission.csv", index=False)

# Vérification du rendu
print(df_submission.head())

Lancement des prédictions sur 14784 surveys...


KeyboardInterrupt: 